In [ ]:
import os

# Force Python path
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Nishanth\anaconda3\envs\pyspark_env\python.exe"

# CRITICAL FIX (network binding)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FINAL_FIX") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "false") \
    .getOrCreate()

In [4]:
data = [("Nishanth", 31), ("John", 28)]
df = spark.createDataFrame(data, ["Name", "Age"])
df.show()

+--------+---+
|    Name|Age|
+--------+---+
|Nishanth| 31|
|    John| 28|
+--------+---+



In [ ]:
#Top 2 Products per Country
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

agg = df.groupBy("country", "product") \
        .agg(sum("amount").alias("total_sales"))

window = Window.partitionBy("country").orderBy(col("total_sales").desc())

result = agg.withColumn("rank", row_number().over(window)) \
            .filter(col("rank") <= 2)

result.show()

In [ ]:
##Top 3 Products by Total Sales
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

# Load data (header handled automatically)
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

# Aggregate total sales per product
total_sales = df.groupBy("product") \
                .agg(sum("amount").alias("total_sales"))

# Sort descending and get top 3
top3 = total_sales.orderBy(col("total_sales").desc()).limit(3)

top3.show()

In [ ]:
#Second Highest Sale Per Country
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Load data
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

# Define window per country ordered by amount descending
window = Window.partitionBy("country").orderBy(col("amount").desc())

# Add rank and filter second highest
second_highest = df.withColumn("rank", row_number().over(window)) \
                   .filter(col("rank") == 2) \
                   .select("country", col("amount").alias("second_highest_sale"))

second_highest.show()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("LogAnalysisDF").getOrCreate()

# Load data
df = spark.read.text("app_logs.txt")

# Split into columns
split_col = split(df["value"], " ")

logs = df.select(
    split_col.getItem(0).alias("date"),
    split_col.getItem(1).alias("time"),
    split_col.getItem(2).alias("level"),
    split_col.getItem(3).alias("user"),
    split_col.getItem(4).alias("action"),
    split_col.getItem(5).alias("status_or_item")
)

logs.show(truncate=False)

In [ ]:
Problem statements
#Count of Each Log Level
#Most Active User
#Failed Login Attempts
#Users with Both Success & Failure
#Consecutive Failures
#Users Who Never Made a Purchase